# Fine-tune DistilBERT on the pandas issue split (Colab Pro, GPU)

Pulls `processed/pandas/{RUN_ID}` from the local MinIO (exposed to Colab via
`ngrok http <minio-port>` — see RUNBOOK.md), fine-tunes
`distilbert-base-uncased`, and pushes `state_dict`, the loss curve, and
`model_card.json` to `artifacts/classifier/distilbert/{RUN_ID}/`.

**Not consumed today.** Nothing in `app/` imports this artifact (FR-020);
the model card's `weights_sha256` pre-positions the Day 2 weights-integrity
check (Rule 4).

In [ ]:
!pip -q install transformers datasets torch scikit-learn boto3 pyarrow pandas matplotlib

In [ ]:
import getpass, os

# ngrok-exposed MinIO endpoint, e.g. https://abc123.ngrok-free.app
MINIO_ENDPOINT = input('MinIO endpoint (ngrok URL): ').strip()
MINIO_ACCESS_KEY = input('MinIO access key (minio_root_user): ').strip()
MINIO_SECRET_KEY = getpass.getpass('MinIO secret key (minio_root_password): ')
RUN_ID = input('Processed run_id to train on: ').strip()
BUCKET = 'maintainers-copilot'
PROCESSED_PREFIX = f'processed/pandas/{RUN_ID}'
ARTIFACT_PREFIX = f'artifacts/classifier/distilbert/{RUN_ID}'

In [ ]:
import boto3, io, pandas as pd

s3 = boto3.client('s3', endpoint_url=MINIO_ENDPOINT,
                  aws_access_key_id=MINIO_ACCESS_KEY,
                  aws_secret_access_key=MINIO_SECRET_KEY)

def read_parquet(split):
    obj = s3.get_object(Bucket=BUCKET, Key=f'{PROCESSED_PREFIX}/{split}.parquet')
    return pd.read_parquet(io.BytesIO(obj['Body'].read()))

train_df, val_df = read_parquet('train'), read_parquet('val')
print('train', train_df.shape, 'val', val_df.shape)
print(train_df['target_class'].value_counts())

In [ ]:
import hashlib, numpy as np
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, f1_score

CLASSES = ['bug', 'feature', 'docs', 'question']
label2id = {c: i for i, c in enumerate(CLASSES)}
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN, EPOCHS, LR, BATCH = 256, 3, 2e-5, 16

def to_ds(df):
    df = df.copy()
    df['text'] = (df['title'].fillna('') + '\n\n' + df['body'].fillna('')).str.slice(0, 4000)
    df['label'] = df['target_class'].map(label2id)
    return Dataset.from_pandas(df[['text', 'label']], preserve_index=False)

# training-data hash over the exact train rows consumed
train_hash = hashlib.sha256(
    pd.util.hash_pandas_object(train_df[['issue_number', 'target_class']], index=False).values.tobytes()
).hexdigest()

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(b): return tok(b['text'], truncation=True, padding='max_length', max_length=MAX_LEN)
train_ds = to_ds(train_df).map(tokenize, batched=True)
val_ds = to_ds(val_df).map(tokenize, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CLASSES),
    id2label={i: c for c, i in label2id.items()}, label2id=label2id)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'macro_f1': f1_score(labels, preds, average='macro')}

args = TrainingArguments(output_dir='out', num_train_epochs=EPOCHS,
    learning_rate=LR, per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH, eval_strategy='epoch',
    logging_steps=20, report_to='none', seed=17)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, compute_metrics=metrics)
trainer.train()
final_val_metrics = trainer.evaluate()
print(final_val_metrics)

In [ ]:
import json, torch, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 1. state_dict -> bytes, sha256 (contract C6)
buf = io.BytesIO(); torch.save(model.state_dict(), buf); weights = buf.getvalue()
weights_sha256 = hashlib.sha256(weights).hexdigest()
s3.put_object(Bucket=BUCKET, Key=f'{ARTIFACT_PREFIX}/state_dict.pt', Body=weights)

# 2. loss curve
hist = trainer.state.log_history
steps = [h['step'] for h in hist if 'loss' in h]
losses = [h['loss'] for h in hist if 'loss' in h]
fig = plt.figure(); plt.plot(steps, losses); plt.xlabel('step'); plt.ylabel('train loss')
lb = io.BytesIO(); fig.savefig(lb, format='png'); plt.close(fig)
s3.put_object(Bucket=BUCKET, Key=f'{ARTIFACT_PREFIX}/loss_curve.png', Body=lb.getvalue())

# 3. model card (data-model.md)
model_card = {
    'architecture': MODEL_NAME,
    'hyperparameters': {'epochs': EPOCHS, 'learning_rate': LR,
                        'batch_size': BATCH, 'max_len': MAX_LEN},
    'training_data_hash': train_hash,
    'final_val_metrics': {'accuracy': final_val_metrics.get('eval_accuracy'),
                          'macro_f1': final_val_metrics.get('eval_macro_f1')},
    'weights_sha256': weights_sha256,
}
s3.put_object(Bucket=BUCKET, Key=f'{ARTIFACT_PREFIX}/model_card.json',
              Body=json.dumps(model_card, indent=2).encode())

# Verify C6: pushed object hash == model_card.weights_sha256
rt = s3.get_object(Bucket=BUCKET, Key=f'{ARTIFACT_PREFIX}/state_dict.pt')['Body'].read()
assert hashlib.sha256(rt).hexdigest() == model_card['weights_sha256'], 'C6 weights hash mismatch'
print('pushed to', ARTIFACT_PREFIX, '\n', json.dumps(model_card, indent=2))